In [5]:
# CELDA 1: IMPORTACIONES Y CONFIGURACIÓN BÁSICA
import requests
import json
import time
import pandas as pd
import boto3
import os
from datetime import datetime, timedelta
from tqdm import tqdm
import sys
import os
# Añadir carpeta padre al PATH
sys.path.append(os.path.dirname(os.getcwd()))

from config import API_KEY

print("✅ Celda 1 ejecutada correctamente")
print(f"   - API_KEY cargada desde config.py (longitud: {len(API_KEY)} caracteres)")
print("   - Librerías importadas: requests, json, time, pandas, boto3, os, datetime, tqdm, psycopg2")

✅ Celda 1 ejecutada correctamente
   - API_KEY cargada desde config.py (longitud: 32 caracteres)
   - Librerías importadas: requests, json, time, pandas, boto3, os, datetime, tqdm, psycopg2


In [6]:
# CELDA 2: CONFIGURACIÓN DE S3 Y CARPETAS LOCALES
import boto3

# Nombre de tu bucket S3 (creado en la consola de AWS)
S3_BUCKET = "rawg-data-lake-manuel-39"

# Cliente de S3
s3_client = boto3.client('s3')

# Ruta local para guardar datos crudos (solo para desarrollo/local)
LOCAL_RAW_DIR = "../data/raw"
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

# Verificar que el bucket S3 exista y sea accesible
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"✅ Bucket S3 '{S3_BUCKET}' accesible.")
except Exception as e:
    print(f"❌ Error al acceder al bucket S3: {e}")
    print("👉 Asegúrate de que el bucket existe en la consola de AWS y que tus credenciales están configuradas.")
    raise

print(f"📁 Directorio local para datos crudos: {os.path.abspath(LOCAL_RAW_DIR)}")

✅ Bucket S3 'rawg-data-lake-manuel-39' accesible.
📁 Directorio local para datos crudos: C:\Users\manue\proyecto_hackaboss\Proyecto_RAWG\rawg-aws-ml-analytics\src\data\raw


In [7]:
# CELDA 3: CONFIGURACIÓN PARA DESCARGA MASIVA HISTÓRICA
# rango: 2016-01-01 → 2026-01-01

from datetime import datetime

BASE_URL = "https://api.rawg.io/api/games"
FECHA_INICIAL = datetime(2016, 1, 1)
FECHA_FINAL = datetime(2026, 1, 1)
DIAS_TOTALES = (FECHA_FINAL - FECHA_INICIAL).days

print("✅ Celda 3 ejecutada correctamente")
print(f"   - URL base: {BASE_URL}")
print(f"   - Rango histórico: {FECHA_INICIAL.date()} → {FECHA_FINAL.date()}")
print(f"   - Total de días: {DIAS_TOTALES} días ({DIAS_TOTALES // 365} años completos)")
print("   - Objetivo: Descargar TODOS los juegos disponibles en este rango histórico")

✅ Celda 3 ejecutada correctamente
   - URL base: https://api.rawg.io/api/games
   - Rango histórico: 2016-01-01 → 2026-01-01
   - Total de días: 3653 días (10 años completos)
   - Objetivo: Descargar TODOS los juegos disponibles en este rango histórico


In [15]:
# CELDA 4: FUNCIÓN PARA EXTRAER JUEGOS POR DÍA (CON UNA SOLA BARRA SI HAY PAGINACIÓN)
def fetch_games_by_date(date_str):
    all_results = []
    page = 1
    
    # Primera llamada para saber cuántas páginas hay
    try:
        response = requests.get(BASE_URL, params={
            "key": API_KEY,
            "dates": date_str,
            "page": 1,
            "page_size": 40
        }, timeout=10)
        response.raise_for_status()
        data = response.json()
        total_pages = min((data.get("count", 0) // 40) + 1, 25)  # Límite de seguridad
        if total_pages == 0:
            return []
    except Exception:
        return []

    # Si hay más de 1 página, mostrar barra
    if total_pages > 1:
        pages_iter = tqdm(range(1, total_pages + 1), desc=f"   📄 {date_str}", leave=False)
    else:
        pages_iter = range(1, total_pages + 1)
    
    for page in pages_iter:
        try:
            response = requests.get(BASE_URL, params={
                "key": API_KEY,
                "dates": date_str,
                "page": page,
                "page_size": 40
            }, timeout=10)
            response.raise_for_status()
            data = response.json()
            results = data.get("results", [])
            all_results.extend(results)
            time.sleep(0.3)
        except Exception:
            time.sleep(1)
            continue
            
    return all_results

# Prueba
test_date = "2023-01-01"
test_games = fetch_games_by_date(test_date)
print(f"✅ Celda 4 lista. Prueba: {len(test_games)} juegos en {test_date}")

✅ Celda 4 lista. Prueba: 1000 juegos en 2023-01-01


In [17]:
# CELDA 5: EXTRACCIÓN MASIVA HISTÓRICA (UNA SOLA BARRA DE PROGRESO)
# Fuente: Adaptado directamente del estilo del profesor Daniel

# Lista para acumular todos los juegos
resultados = []

print("🔄 Iniciando extracción masiva histórica...")
print(f"   - Rango: {FECHA_INICIAL.date()} → {FECHA_FINAL.date()}")
print(f"   - Total de días: {DIAS_TOTALES}")

# Bucle principal con tqdm (una sola barra)
for i in tqdm(range(DIAS_TOTALES), desc="📅 Descargando datos históricos", unit="día"):
    fecha_actual = FECHA_INICIAL + timedelta(days=i)
    fecha_str = fecha_actual.strftime("%Y-%m-%d")
    
    # Parámetros para la API de RAWG
    params = {
        "key": API_KEY,
        "page_size": 100,  # Máximo permitido por RAWG
        "dates": f"{fecha_str},{fecha_str}"  # Filtrar por fecha exacta
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Añadir resultados si existen
        if "results" in data and data["results"]:
            resultados.extend(data["results"])
            
        # Respetar rate limit de la API
        time.sleep(0.1)
        
    except Exception as e:
        # Continuar con el siguiente día si hay error (silencioso)
        continue

# Guardar resultados en archivo JSON local
archivo_local = "extraccion_historica.json"
with open(archivo_local, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print(f"\n✅ Extracción masiva histórica completada")
print(f"   - Juegos descargados: {len(resultados)}")
print(f"   - Guardando en archivo local: {archivo_local}")
print("   - Archivo guardado: extraccion_historica.json")
print("   - ✅ Listo para subir a S3 en la siguiente fase")

🔄 Iniciando extracción masiva histórica...
   - Rango: 2016-01-01 → 2026-01-01
   - Total de días: 3653


📅 Descargando datos históricos: 100%|██████████| 3653/3653 [3:23:16<00:00,  3.34s/día]      



✅ Extracción masiva histórica completada
   - Juegos descargados: 133295
   - Guardando en archivo local: extraccion_historica.json
   - Archivo guardado: extraccion_historica.json
   - ✅ Listo para subir a S3 en la siguiente fase


In [18]:
# CELDA 6: SUBIR ARCHIVO HISTÓRICO A S3
# Fuente: Adaptado directamente de 17.AWS - S3.txt
# Objetivo: Subir 'extraccion_historica.json' al bucket S3 en carpeta 'raw/'

import boto3
import os

# Configuración
S3_BUCKET_NAME = "rawg-data-lake-manuel-39"
LOCAL_FILE = "extraccion_historica.json"
S3_KEY = "raw/extraccion_historica.json"

# Verificar que el archivo local exista
if not os.path.exists(LOCAL_FILE):
    raise FileNotFoundError(f"❌ Archivo local no encontrado: {LOCAL_FILE}")

# Inicializar cliente S3
s3_client = boto3.client('s3')

print("📤 Iniciando subida a S3...")
print(f"   - Archivo local: {LOCAL_FILE}")
print(f"   - Bucket S3: {S3_BUCKET_NAME}")
print(f"   - Ruta en S3: {S3_KEY}")

try:
    # Subir archivo a S3
    s3_client.upload_file(
        Filename=LOCAL_FILE,
        Bucket=S3_BUCKET_NAME,
        Key=S3_KEY
    )
    print("\n✅ Archivo subido exitosamente a S3")
    print(f"   - Ubicación: s3://{S3_BUCKET_NAME}/{S3_KEY}")
    print("   - ✅ Listo para la siguiente fase: carga a RDS")
    
except Exception as e:
    print(f"\n❌ Error al subir a S3: {e}")
    raise

📤 Iniciando subida a S3...
   - Archivo local: extraccion_historica.json
   - Bucket S3: rawg-data-lake-manuel-39
   - Ruta en S3: raw/extraccion_historica.json

✅ Archivo subido exitosamente a S3
   - Ubicación: s3://rawg-data-lake-manuel-39/raw/extraccion_historica.json
   - ✅ Listo para la siguiente fase: carga a RDS


In [21]:
# CELDA 7: LEER ARCHIVO HISTÓRICO DESDE S3 Y CARGAR EN DATAFRAME
# Fuente: Estilo del profesor Daniel — exploración antes de cargar a base de datos

import boto3
import json
import pandas as pd
import os

# Configuración
S3_BUCKET_NAME = "rawg-data-lake-manuel-39"
S3_KEY = "raw/extraccion_historica.json"

# Cliente S3
s3_client = boto3.client('s3')

print("🔍 Leyendo archivo histórico desde S3...")
try:
    # Descargar el objeto como bytes
    response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=S3_KEY)
    body = response['Body'].read()
    
    # Convertir a lista de diccionarios
    data = json.loads(body)
    
    if isinstance(data, list):
        df = pd.json_normalize(data)
        print(f"✅ DataFrame cargado exitosamente")
        print(f"   - Filas: {len(df)} | Columnas: {len(df.columns)}")
        print(f"   - Columnas principales: {list(df.columns)[:12]}")
        
        # Mostrar primeras filas
        print("\n📊 Primeras 3 filas del DataFrame:")
        print(df.head(3))
        
        # 👇 CREAR CARPETA ANTES DE GUARDAR
        LOCAL_DATA_DIR = "data/raw"
        os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
        csv_path = os.path.join(LOCAL_DATA_DIR, "extraccion_historica.csv")
        
        # Guardar copia local
        df.to_csv(csv_path, index=False)
        print(f"   - Guardado como CSV local: {csv_path}")
        
    else:
        print(f"⚠️ El archivo no es una lista. Tipo: {type(data)}")
        
except Exception as e:
    print(f"❌ Error al leer desde S3: {e}")
    raise

🔍 Leyendo archivo histórico desde S3...
✅ DataFrame cargado exitosamente
   - Filas: 133295 | Columnas: 42
   - Columnas principales: ['slug', 'name', 'playtime', 'platforms', 'stores', 'released', 'tba', 'background_image', 'rating', 'rating_top', 'ratings', 'ratings_count']

📊 Primeras 3 filas del DataFrame:
                                 slug                                name  \
0  princess-remedy-in-a-world-of-hurt  Princess Remedy in a World of Hurt   
1                                echo                                ECHO   
2                            murnatan                            Murnatan   

   playtime                                          platforms  \
0         1  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
1         2  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
2         1  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   

                                              stores    released    tba  \
0  [{'store': {'id': 1, 'name': 'Steam'

In [60]:
# CELDA 8: LAMBDA 2 — EXTRACCIÓN INCREMENTAL DIARIA (7 archivos separados)
# Fuente: Indicaciones del profesor Daniel
# Propósito: Crear UN ARCHIVO POR DÍA en S3 (simula trigger diario)

import requests
import json
import boto3
from datetime import datetime, timedelta

# 🔑 Configuración
API_KEY = "53721c9ba8a74e818865f9bb81ccb779"
S3_BUCKET = "rawg-data-lake-manuel-39"

print("🔄 Iniciando extracción incremental diaria...")
print(f"   - Bucket S3: {S3_BUCKET}")
print(f"   - Rango: últimos 7 días\n")

# Paso 1: Calcular fechas (del 1 al 7 de febrero)
today = datetime.now()
dates_to_extract = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(7)]
dates_to_extract.reverse()  # Del más antiguo al más reciente

print("📅 Fechas a extraer:")
for i, date in enumerate(dates_to_extract, 1):
    print(f"   {i}. {date}")

# Paso 2: Extraer juegos para CADA día
total_files_created = 0

for date in dates_to_extract:
    print(f"\n📥 Extrayendo juegos para: {date}")
    
    # Consultar API de RAWG para este día específico
    params = {
        "key": API_KEY,
        "dates": f"{date},{date}",
        "page_size": 20,
        "ordering": "-released"
    }
    
    try:
        response = requests.get("https://api.rawg.io/api/games", params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        games = data.get("results", [])
        
        print(f"   ✅ Encontrados {len(games)} juegos")
        
        # Guardar en S3 SOLO si hay juegos
        if games:
            payload = {
                "extraction_date": date,
                "games_count": len(games),
                "games": games[:10]  # Límite para no exceder quota
            }
            
            try:
                s3 = boto3.client('s3')
                s3_key = f"raw/incremental/juegos_incremental_{date}.json"
                s3.put_object(
                    Bucket=S3_BUCKET,
                    Key=s3_key,
                    Body=json.dumps(payload, indent=2, ensure_ascii=False),
                    ContentType='application/json'
                )
                print(f"   📤 Archivo guardado: {s3_key}")
                total_files_created += 1
                
            except Exception as e:
                print(f"   ❌ Error al guardar en S3: {e}")
        else:
            print(f"   ℹ️ No hay juegos nuevos para {date}")
            
    except Exception as e:
        print(f"   ⚠️ Error al consultar API para {date}: {e}")

# Resumen final
print("\n" + "="*60)
print("🎉 EXTRACCIÓN INCREMENTAL COMPLETADA")
print("="*60)
print(f"✅ Archivos creados en S3: {total_files_created}")
print(f"📁 Ubicación: s3://{S3_BUCKET}/raw/incremental/")
print(f"\n💡 Nota: Cada archivo activaría el trigger S3 → Lambda 3")
print(f"   en producción (uno por día).")
print("="*60)

🔄 Iniciando extracción incremental diaria...
   - Bucket S3: rawg-data-lake-manuel-39
   - Rango: últimos 7 días

📅 Fechas a extraer:
   1. 2026-02-01
   2. 2026-02-02
   3. 2026-02-03
   4. 2026-02-04
   5. 2026-02-05
   6. 2026-02-06
   7. 2026-02-07

📥 Extrayendo juegos para: 2026-02-01
   ✅ Encontrados 1 juegos
   📤 Archivo guardado: raw/incremental/juegos_incremental_2026-02-01.json

📥 Extrayendo juegos para: 2026-02-02
   ✅ Encontrados 1 juegos
   📤 Archivo guardado: raw/incremental/juegos_incremental_2026-02-02.json

📥 Extrayendo juegos para: 2026-02-03
   ✅ Encontrados 0 juegos
   ℹ️ No hay juegos nuevos para 2026-02-03

📥 Extrayendo juegos para: 2026-02-04
   ✅ Encontrados 3 juegos
   📤 Archivo guardado: raw/incremental/juegos_incremental_2026-02-04.json

📥 Extrayendo juegos para: 2026-02-05
   ✅ Encontrados 4 juegos
   📤 Archivo guardado: raw/incremental/juegos_incremental_2026-02-05.json

📥 Extrayendo juegos para: 2026-02-06
   ✅ Encontrados 1 juegos
   📤 Archivo guardado: r

In [46]:
# # CELDA 8: LAMBDA 2 — EXTRACCIÓN INCREMENTAL (últimos 7 días)
# import requests
# import json
# import boto3
# from datetime import datetime, timedelta

# API_KEY = "53721c9ba8a74e818865f9bb81ccb779"
# S3_BUCKET = "rawg-data-lake-manuel-39"

# # Fechas: últimos 7 días (garantiza encontrar juegos)
# end_date = datetime.now().strftime("%Y-%m-%d")
# start_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

# print(f"🔄 Extrayendo juegos de los últimos 7 días ({start_date} a {end_date})...")

# params = {
#     "key": API_KEY,
#     "dates": f"{start_date},{end_date}",
#     "page_size": 40,
#     "ordering": "-released"
# }

# response = requests.get("https://api.rawg.io/api/games", params=params)
# games = response.json().get("results", [])

# print(f"✅ Encontrados {len(games)} juegos en los últimos 7 días")

# if games:
#     payload = {
#         "extraction_date": end_date,
#         "date_range": f"{start_date} to {end_date}",
#         "games_count": len(games),
#         "games": games[:20]  # Limitar a 20 para no exceder quota
#     }
    
#     s3 = boto3.client('s3')
#     s3_key = f"raw/incremental/games_incremental_{end_date}_7days.json"
#     s3.put_object(
#         Bucket=S3_BUCKET,
#         Key=s3_key,
#         Body=json.dumps(payload, indent=2, ensure_ascii=False),
#         ContentType='application/json'
#     )
#     print(f"📤 Guardado en S3: s3://{S3_BUCKET}/{s3_key}")
#     print("\n🎉 Lambda 2 demostrada con datos reales")
# else:
#     print("⚠️ No se encontraron juegos (improbable con 7 días)")

🔄 Extrayendo juegos de los últimos 7 días (2026-01-31 a 2026-02-07)...
✅ Encontrados 10 juegos en los últimos 7 días
📤 Guardado en S3: s3://rawg-data-lake-manuel-39/raw/incremental/games_incremental_2026-02-07_7days.json

🎉 Lambda 2 demostrada con datos reales


In [47]:
df

,id,name,slug,released,tba,rating,metacritic,playtime,background_image,success
0,18418,Princess Remedy in a World of Hurt,princess-remedy-in-a-world-of-hurt,2016-01-01,False,2.85,NaN,1,https://media.rawg.io/media/screenshots/138/13...,0
1,43712,ECHO,echo,2016-01-01,False,3.60,76.0,2,https://media.rawg.io/media/games/584/58478387...,0
2,43650,Murnatan,murnatan,2016-01-01,False,0.00,NaN,1,https://media.rawg.io/media/screenshots/049/04...,0
3,43032,Objects In Space,objects-in-space,2016-01-01,False,3.57,NaN,2,https://media.rawg.io/media/screenshots/6ac/6a...,0
4,42058,Rainbow Skies,rainbow-skies,2016-01-01,False,0.00,NaN,0,https://media.rawg.io/media/screenshots/366/36...,0
...,...,...,...,...,...,...,...,...,...,...
133290,973544,MОUSE,mouse-3,2025-12-31,False,3.17,NaN,0,https://media.rawg.io/media/screenshots/f47/f4...,0
133291,984637,Cairn (2025),cairn-2025,2025-12-31,False,0.00,NaN,0,https://media.rawg.io/media/games/147/1474adaa...,0
133292,1015555,AI Games,ai-games,2025-12-31,False,0.00,NaN,0,https://media.rawg.io/media/screenshots/312/31...,0
133293,1015546,FNAF Free,fnaf-free,2025-12-31,False,0.00,NaN,0,NaN,0


In [1]:
# CELDA 8: DEFINIR MÉTRICA DE ÉXITO Y PREPARAR DATOS PARA RDS
# Fuente: Estilo del profesor Daniel — minimalista y funcional

import pandas as pd
import os

# Cargar el CSV desde data/raw/
csv_path = "data/raw/extraccion_historica.csv"
df = pd.read_csv(csv_path)

print("📊 Celda 8: Preparación de datos para RDS")
print(f"   - DataFrame original: {len(df)} filas, {len(df.columns)} columnas")

# Paso 1: Seleccionar columnas relevantes (solo las que vamos a usar en RDS)
cols_interes = [
    'id', 'name', 'slug', 'released', 'tba',
    'rating', 'metacritic', 'playtime',
    'background_image', 'website'
]
# Asegurarse de que existan (RAWG puede variar ligeramente)
cols_existentes = [col for col in cols_interes if col in df.columns]
df_clean = df[cols_existentes].copy()

# Paso 2: Definir métrica de éxito (simple y numérica)
df_clean['success'] = (df_clean['rating'] >= 4.0).astype(int)

# Paso 3: Limpieza básica
df_clean['released'] = pd.to_datetime(df_clean['released'], errors='coerce')
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')
df_clean['metacritic'] = pd.to_numeric(df_clean['metacritic'], errors='coerce')

print(f"   - Columnas seleccionadas: {list(df_clean.columns)}")
print(f"   - Juegos con success=1 (rating ≥ 4.0): {df_clean['success'].sum()} ({df_clean['success'].mean():.1%})")
print(f"   - Valores nulos en 'rating': {df_clean['rating'].isna().sum()}")

# Guardar versión limpia para RDS
output_path = "data/processed/games_for_rds.csv"
os.makedirs("data/processed", exist_ok=True)
df_clean.to_csv(output_path, index=False)
print(f"✅ Datos preparados guardados en: {output_path}")

📊 Celda 8: Preparación de datos para RDS
   - DataFrame original: 133295 filas, 42 columnas
   - Columnas seleccionadas: ['id', 'name', 'slug', 'released', 'tba', 'rating', 'metacritic', 'playtime', 'background_image', 'success']
   - Juegos con success=1 (rating ≥ 4.0): 1556 (1.2%)
   - Valores nulos en 'rating': 0
✅ Datos preparados guardados en: data/processed/games_for_rds.csv


In [12]:
# Prueba
import psycopg2
try:
    conn = psycopg2.connect(
        host="rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com",
        port=5432,
        dbname="postgres",
        user="adminuser",
        password="Admin123456!"
    )
    print("🎉 ¡CONEXIÓN EXITOSA! RDS ya es accesible desde tu PC.")
    conn.close()
except Exception as e:
    print("❌ Error:", str(e)[:60])

🎉 ¡CONEXIÓN EXITOSA! RDS ya es accesible desde tu PC.


In [15]:
# CELDA 9: CREACIÓN DEL ESQUEMA RELACIONAL MÍNIMO EN RDS
# Fuente: Adaptado 100% de 01_create_rawg_database_optimized.sql
# Objetivo: Implementar solo las tablas necesarias para el modelo de ML

import psycopg2
from psycopg2 import errors

# 🔑 Configuración de RDS
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

print("🚀 Iniciando creación del esquema relacional mínimo en RDS...")
print(f"   - Host: {RDS_HOST}")
print(f"   - Base de datos: {RDS_DBNAME}")

# Paso 1: Crear base de datos 'rawg_games_db' (sin IF NOT EXISTS)
try:
    conn = psycopg2.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        dbname="postgres",
        user=RDS_USER,
        password=RDS_PASSWORD
    )
    conn.autocommit = True
    cur = conn.cursor()
    
    # PostgreSQL no soporta "IF NOT EXISTS" en CREATE DATABASE
    try:
        cur.execute("CREATE DATABASE rawg_games_db;")
        print("✅ Base de datos 'rawg_games_db' creada")
    except errors.DuplicateDatabase:
        print("ℹ️ Base de datos 'rawg_games_db' ya existe")
    
    conn.close()
except Exception as e:
    print(f"❌ Error al crear base de datos: {e}")
    raise

# Paso 2: Conectarse a 'rawg_games_db'
conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)
conn.autocommit = True
cur = conn.cursor()

# Paso 3: Crear tablas maestras (3)
cur.execute("""
CREATE TABLE IF NOT EXISTS esrb_ratings (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);
""")
print("✅ Tabla 'esrb_ratings' creada")

cur.execute("""
CREATE TABLE IF NOT EXISTS genres (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);
""")
print("✅ Tabla 'genres' creada")

cur.execute("""
CREATE TABLE IF NOT EXISTS platforms (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);
""")
print("✅ Tabla 'platforms' creada")

# Paso 4: Crear tabla principal 'games' (1)
cur.execute("""
CREATE TABLE IF NOT EXISTS games (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    released DATE,
    rating NUMERIC(3,2),
    ratings_count INTEGER,
    metacritic INTEGER,
    playtime INTEGER,
    status_yet INTEGER DEFAULT 0,
    status_owned INTEGER DEFAULT 0,
    status_beaten INTEGER DEFAULT 0,
    status_toplay INTEGER DEFAULT 0,
    status_dropped INTEGER DEFAULT 0,
    status_playing INTEGER DEFAULT 0,
    success BOOLEAN DEFAULT FALSE,
    esrb_rating_id INTEGER REFERENCES esrb_ratings(id) ON DELETE SET NULL,
    updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
print("✅ Tabla 'games' creada")

# Paso 5: Tablas relacionales N:M (2)
cur.execute("""
CREATE TABLE IF NOT EXISTS game_genres (
    game_id INTEGER NOT NULL REFERENCES games(id) ON DELETE CASCADE,
    genre_id INTEGER NOT NULL REFERENCES genres(id) ON DELETE CASCADE,
    PRIMARY KEY (game_id, genre_id)
);
""")
print("✅ Tabla 'game_genres' creada")

cur.execute("""
CREATE TABLE IF NOT EXISTS game_platforms (
    game_id INTEGER NOT NULL REFERENCES games(id) ON DELETE CASCADE,
    platform_id INTEGER NOT NULL REFERENCES platforms(id) ON DELETE CASCADE,
    released_at DATE,
    PRIMARY KEY (game_id, platform_id)
);
""")
print("✅ Tabla 'game_platforms' creada")

# Paso 6: Insertar valores iniciales en esrb_ratings
cur.execute("""
INSERT INTO esrb_ratings (id, name) VALUES
(1, 'Everyone'),
(2, 'Everyone 10+'),
(3, 'Teen'),
(4, 'Mature'),
(5, 'Adults Only'),
(6, 'Rating Pending')
ON CONFLICT (id) DO NOTHING;
""")
print("✅ Valores iniciales insertados en 'esrb_ratings'")

# Paso 7: Crear función calculate_success()
cur.execute("""
CREATE OR REPLACE FUNCTION calculate_success()
RETURNS VOID AS $$
BEGIN
    UPDATE games
    SET success = CASE
        WHEN rating >= 4.0 AND ratings_count >= 1000 THEN TRUE
        ELSE FALSE
    END
    WHERE rating IS NOT NULL AND ratings_count IS NOT NULL;
END;
$$ LANGUAGE plpgsql;
""")
print("✅ Función 'calculate_success()' creada")

# Paso 8: Crear vista games_for_ml
cur.execute("""
CREATE OR REPLACE VIEW games_for_ml AS
SELECT
    g.id,
    g.name,
    EXTRACT(YEAR FROM g.released) AS release_year,
    g.rating,
    g.ratings_count,
    g.metacritic,
    g.playtime,
    g.status_yet,
    g.status_owned,
    g.status_beaten,
    g.status_toplay,
    g.status_dropped,
    g.status_playing,
    g.success,
    er.name AS esrb_rating
FROM games g
LEFT JOIN esrb_ratings er ON g.esrb_rating_id = er.id;
""")
print("✅ Vista 'games_for_ml' creada (sin STRING_AGG para simplificar)")

# Cierre
cur.close()
conn.close()
print("\n🎉 ¡Esquema relacional mínimo creado en RDS!")
print("   - 6 tablas implementadas")
print("   - Función y vista para ML listas")
print("   - Listo para cargar datos y entrenar XGBoost")

🚀 Iniciando creación del esquema relacional mínimo en RDS...
   - Host: rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com
   - Base de datos: rawg_games_db
✅ Base de datos 'rawg_games_db' creada
✅ Tabla 'esrb_ratings' creada
✅ Tabla 'genres' creada
✅ Tabla 'platforms' creada
✅ Tabla 'games' creada
✅ Tabla 'game_genres' creada
✅ Tabla 'game_platforms' creada
✅ Valores iniciales insertados en 'esrb_ratings'
✅ Función 'calculate_success()' creada
✅ Vista 'games_for_ml' creada (sin STRING_AGG para simplificar)

🎉 ¡Esquema relacional mínimo creado en RDS!
   - 6 tablas implementadas
   - Función y vista para ML listas
   - Listo para cargar datos y entrenar XGBoost


In [36]:
# CELDA DE DIAGNÓSTICO: Ver columnas reales del CSV
import pandas as pd

df_test = pd.read_csv("data/processed/games_for_rds.csv", nrows=5)
print("Columnas en el CSV:")
print(df_test.columns.tolist())

Columnas en el CSV:
['id', 'name', 'slug', 'released', 'tba', 'rating', 'metacritic', 'playtime', 'background_image', 'success']


In [38]:
# CELDA LIMPIEZA: Vaciar solo las tablas que existen (6 tablas)
import psycopg2

RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

print("🧹 Iniciando limpieza de tablas...")

conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)
conn.autocommit = True
cur = conn.cursor()

# Solo las tablas que SÍ existen (en orden inverso para FK)
tablas_a_vaciar = [
    "game_platforms",   # depende de games y platforms
    "game_genres",      # depende de games y genres
    "games"             # tabla principal
]

for tabla in tablas_a_vaciar:
    print(f"   → Vaciamos '{tabla}'...", end="")
    cur.execute(f"DELETE FROM {tabla};")
    print(" ✅")

# NOTA: No vaciamos esrb_ratings, genres, platforms porque contienen datos maestros
# (y no queremos perder los valores iniciales como 'Everyone', 'Teen', etc.)

cur.close()
conn.close()
print("✅ Limpieza completada. Tablas vacías y listas para recarga.")

🧹 Iniciando limpieza de tablas...
   → Vaciamos 'game_platforms'... ✅
   → Vaciamos 'game_genres'... ✅
 ✅ → Vaciamos 'games'...
✅ Limpieza completada. Tablas vacías y listas para recarga.


In [48]:
# # --- Paso 1: Cargar 'games' en lotes ---
# print("💾 Paso 1/4: Insertando en 'games' (por lotes)...")

# def chunker(seq, size):
#     return (seq[pos:pos + size] for pos in range(0, len(seq), size))

# total_inserted = 0
# for batch in chunker(games_data, 1000):  # Lotes de 1000 filas
#     cur.executemany("""
#         INSERT INTO games (id, name, released, rating, metacritic, playtime, success)
#         VALUES (%s, %s, %s, %s, %s, %s, %s)
#         ON CONFLICT (id) DO UPDATE SET
#             name = EXCLUDED.name,
#             released = EXCLUDED.released,
#             rating = EXCLUDED.rating,
#             metacritic = EXCLUDED.metacritic,
#             playtime = EXCLUDED.playtime,
#             success = EXCLUDED.success;
#     """, batch)
#     total_inserted += len(batch)
#     print(f"   → {total_inserted} / {len(games_data)} juegos insertados")

# print(f"✅ {total_inserted} juegos insertados en 'games'")

In [39]:
# CELDA 10: CARGA SEGURA POR LOTES EN LAS 6 TABLAS DE RDS
# Fuente: Adaptado 100% de 01_create_rawg_database_optimized.sql

import pandas as pd
import psycopg2
import os
from datetime import datetime

# 🔑 Configuración
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"
CSV_PATH = "data/processed/games_for_rds.csv"

print("📥 Iniciando carga segura en RDS (6 tablas)...")

# --- Paso 0: Verificar CSV ---
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ Archivo no encontrado: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
print(f"✅ CSV cargado: {len(df)} filas")

# --- Preparar datos ---
games_data = []
for _, row in df.iterrows():
    released = None
    if pd.notna(row['released']):
        try:
            released = datetime.strptime(str(row['released']), '%Y-%m-%d').date()
        except:
            pass
    games_data.append((
        int(row['id']),
        str(row['name']),
        released,
        float(row['rating']) if pd.notna(row['rating']) else None,
        int(row['metacritic']) if pd.notna(row['metacritic']) else None,
        int(row['playtime']) if pd.notna(row['playtime']) else None,
        bool(row['success'])
    ))

# --- Función para cargar por lotes ---
def load_in_batches():
    conn = psycopg2.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        dbname=RDS_DBNAME,
        user=RDS_USER,
        password=RDS_PASSWORD
    )
    conn.autocommit = True
    cur = conn.cursor()

    # Paso 1: Cargar 'games'
    print("💾 Paso 1/4: Insertando en 'games' (lotes de 1000)...")
    total = len(games_data)
    for i in range(0, total, 1000):
        batch = games_data[i:i+1000]
        cur.executemany("""
            INSERT INTO games (id, name, released, rating, metacritic, playtime, success)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO UPDATE SET
                name = EXCLUDED.name,
                released = EXCLUDED.released,
                rating = EXCLUDED.rating,
                metacritic = EXCLUDED.metacritic,
                playtime = EXCLUDED.playtime,
                success = EXCLUDED.success;
        """, batch)
        print(f"   → {min(i+1000, total)} / {total} juegos insertados")
    
    # Paso 2: Géneros y plataformas
    print("🔍 Paso 2/4: Asegurando datos maestros...")
    cur.execute("INSERT INTO genres (id, name) VALUES (999, 'Unknown') ON CONFLICT DO NOTHING;")
    cur.execute("INSERT INTO platforms (id, name) VALUES (999, 'PC') ON CONFLICT DO NOTHING;")

    # Paso 3: Relaciones
    print("🔗 Paso 3/4: Creando relaciones N:M...")
    game_ids = [(row[0],) for row in games_data]  # row[0] = id
    for i in range(0, len(game_ids), 1000):
        batch = game_ids[i:i+1000]
        cur.executemany("INSERT INTO game_genres (game_id, genre_id) VALUES (%s, 999) ON CONFLICT DO NOTHING;", batch)
        cur.executemany("INSERT INTO game_platforms (game_id, platform_id) VALUES (%s, 999) ON CONFLICT DO NOTHING;", batch)

    # Paso 4: Métrica
    print("🎯 Paso 4/4: Actualizando métrica de éxito...")
    cur.execute("UPDATE games SET success = (rating >= 4.0) WHERE rating IS NOT NULL;")

    cur.close()
    conn.close()
    print("✅ Conexión cerrada correctamente")

# Ejecutar
load_in_batches()
print("\n🎉 ¡Carga completada en las 6 tablas!")

📥 Iniciando carga segura en RDS (6 tablas)...
✅ CSV cargado: 133295 filas
💾 Paso 1/4: Insertando en 'games' (lotes de 1000)...
   → 1000 / 133295 juegos insertados
   → 2000 / 133295 juegos insertados
   → 3000 / 133295 juegos insertados
   → 4000 / 133295 juegos insertados
   → 5000 / 133295 juegos insertados
   → 6000 / 133295 juegos insertados
   → 7000 / 133295 juegos insertados
   → 8000 / 133295 juegos insertados
   → 9000 / 133295 juegos insertados
   → 10000 / 133295 juegos insertados
   → 11000 / 133295 juegos insertados
   → 12000 / 133295 juegos insertados
   → 13000 / 133295 juegos insertados
   → 14000 / 133295 juegos insertados
   → 15000 / 133295 juegos insertados
   → 16000 / 133295 juegos insertados
   → 17000 / 133295 juegos insertados
   → 18000 / 133295 juegos insertados
   → 19000 / 133295 juegos insertados
   → 20000 / 133295 juegos insertados
   → 21000 / 133295 juegos insertados
   → 22000 / 133295 juegos insertados
   → 23000 / 133295 juegos insertados
   → 240

OperationalError: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.


In [40]:
# CELDA DE VERIFICACIÓN
import psycopg2

RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM games;")
count = cur.fetchone()[0]
print(f"📊 Filas en 'games': {count}")
cur.close()
conn.close()

📊 Filas en 'games': 133295


In [42]:
# CELDA DE CORRECCIÓN (solo relaciones N:M)
import psycopg2

RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"  # ← ¡Faltaba el signo '=' aquí!

conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)
cur = conn.cursor()

print("🔗 Insertando relaciones N:M...")
cur.execute("""
    INSERT INTO game_genres (game_id, genre_id)
    SELECT id, 999 FROM games
    ON CONFLICT DO NOTHING;
""")

cur.execute("""
    INSERT INTO game_platforms (game_id, platform_id)
    SELECT id, 999 FROM games
    ON CONFLICT DO NOTHING;
""")

print("✅ Relaciones N:M actualizadas")
cur.close()
conn.close()

🔗 Insertando relaciones N:M...
✅ Relaciones N:M actualizadas


In [65]:
# CELDA 11: ENTRENAMIENTO DE XGBOOST DESDE RDS
# Fuente: Adaptado de diagrama_Indicaciones del proyecto.txt
# Objetivo: Entrenar modelo usando la vista 'games_for_ml' en RDS

import pandas as pd
import psycopg2
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import joblib
import os
import time

# 🔑 Configuración de RDS
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

print("🧠 Iniciando entrenamiento de XGBoost desde RDS...")
print(f"   - Base de datos: {RDS_DBNAME}")
print(f"   - Host: {RDS_HOST}")

# Paso 1: Conectar y cargar datos desde la vista 'games_for_ml'
start_time = time.time()
try:
    conn = psycopg2.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        dbname=RDS_DBNAME,
        user=RDS_USER,
        password=RDS_PASSWORD
    )
    # Consulta optimizada (solo columnas numéricas para ML)
    query = """
    SELECT 
        rating,
        metacritic,
        playtime,
        success
    FROM games_for_ml
    WHERE rating IS NOT NULL 
      AND success IS NOT NULL;
    """
    df_ml = pd.read_sql(query, conn)
    conn.close()
    
    elapsed = time.time() - start_time
    print(f"✅ Datos cargados desde RDS: {len(df_ml)} filas en {elapsed:.1f} segundos")
    print(f"   - Distribución de 'success': {df_ml['success'].value_counts().to_dict()}")
    
except Exception as e:
    print(f"❌ Error al cargar datos de RDS: {e}")
    raise

# Paso 2: Preparar características y objetivo
X = df_ml[['rating', 'metacritic', 'playtime']]
y = df_ml['success'].astype(int)

print(f"\n📊 Características para el modelo: {list(X.columns)}")
print(f"🎯 Target: 'success' (0 = no exitoso, 1 = exitoso)")

# Paso 3: Dividir datos (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"   - Train: {len(X_train)} filas | Test: {len(X_test)} filas")

# Paso 4: Entrenar modelo XGBoost
print("\n🚀 Entrenando modelo XGBoost...")
model_start = time.time()
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
model.fit(X_train, y_train)
model_time = time.time() - model_start
print(f"✅ Modelo entrenado en {model_time:.1f} segundos")

# Paso 5: Evaluar modelo
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print(f"\n📈 Resultados del modelo:")
print(f"   - Accuracy: {acc:.4f}")
print(f"   - ROC-AUC:  {roc:.4f}")
print(f"   - Clases en test: {y_test.value_counts().to_dict()}")

# Mostrar reporte detallado
if len(y_test.unique()) > 1:
    print("\n📋 Reporte de clasificación:")
    print(classification_report(y_test, y_pred, target_names=['No exitoso', 'Exitoso']))

# Paso 6: Guardar modelo
os.makedirs("models", exist_ok=True)
model_path = "models/xgb_success_model.pkl"
joblib.dump(model, model_path)
print(f"\n📦 Modelo guardado en: {model_path}")

# Paso 7: Ejemplo de predicción
sample_idx = 0
sample = X_test.iloc[sample_idx].values.reshape(1, -1)
pred = model.predict(sample)[0]
prob = model.predict_proba(sample)[0][1]

print(f"\n🔍 Ejemplo de predicción (fila #{sample_idx}):")
print(f"   - Entrada: rating={sample[0][0]:.1f}, metacritic={sample[0][1]:.0f}, playtime={sample[0][2]:.0f}")
print(f"   - Predicción: {'✅ Exitoso' if pred == 1 else '❌ No exitoso'} (probabilidad = {prob:.2%})")

total_time = time.time() - start_time
print(f"\n🎉 ¡Entrenamiento completado en {total_time:.1f} segundos!")
print("   - Modelo listo para FastAPI")

🧠 Iniciando entrenamiento de XGBoost desde RDS...
   - Base de datos: rawg_games_db
   - Host: rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com


C:\Users\manue\AppData\Local\Temp\ipykernel_20296\1026244512.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ml = pd.read_sql(query, conn)


✅ Datos cargados desde RDS: 133295 filas en 6.1 segundos
   - Distribución de 'success': {False: 131739, True: 1556}

📊 Características para el modelo: ['rating', 'metacritic', 'playtime']
🎯 Target: 'success' (0 = no exitoso, 1 = exitoso)
   - Train: 106636 filas | Test: 26659 filas

🚀 Entrenando modelo XGBoost...


C:\ProgramData\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [05:43:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Modelo entrenado en 0.4 segundos

📈 Resultados del modelo:
   - Accuracy: 1.0000
   - ROC-AUC:  1.0000
   - Clases en test: {0: 26348, 1: 311}

📋 Reporte de clasificación:
              precision    recall  f1-score   support

  No exitoso       1.00      1.00      1.00     26348
     Exitoso       1.00      1.00      1.00       311

    accuracy                           1.00     26659
   macro avg       1.00      1.00      1.00     26659
weighted avg       1.00      1.00      1.00     26659


📦 Modelo guardado en: models/xgb_success_model.pkl

🔍 Ejemplo de predicción (fila #0):
   - Entrada: rating=0.0, metacritic=nan, playtime=0
   - Predicción: ❌ No exitoso (probabilidad = 0.00%)

🎉 ¡Entrenamiento completado en 6.8 segundos!
   - Modelo listo para FastAPI


In [49]:
# CELDA 12A: CONFIGURACIÓN PARA ANACONDA
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio aplicado — FastAPI funcionará en Anaconda/Jupyter")

✅ nest_asyncio aplicado — FastAPI funcionará en Anaconda/Jupyter


In [50]:
# CELDA 12B: FASTAPI MÍNIMA (obligatoria para entrega)
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import joblib
import psycopg2
import pandas as pd
import threading
import uvicorn
import os

# Configuración
MODEL_PATH = "models/xgb_success_model.pkl"
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

# Cargar modelo
print("📦 Cargando modelo XGBoost...")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Modelo no encontrado: {MODEL_PATH}")
model = joblib.load(MODEL_PATH)
print("✅ Modelo cargado")

# Crear app
app = FastAPI(title="RAWG Games Success Predictor API", version="1.0.0")

# Modelos Pydantic
class GameFeatures(BaseModel):
    rating: float
    metacritic: Optional[float] = None
    playtime: Optional[int] = None

# Endpoint /health
@app.get("/health")
def health():
    try:
        conn = psycopg2.connect(
            host=RDS_HOST, port=RDS_PORT, dbname=RDS_DBNAME,
            user=RDS_USER, password=RDS_PASSWORD
        )
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM games;")
        count = cur.fetchone()[0]
        cur.close()
        conn.close()
        return {
            "status": "healthy",
            "database": "connected",
            "games_count": count,
            "model_loaded": True
        }
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Database error: {str(e)}")

# Endpoint /predict
@app.post("/predict")
def predict(features: GameFeatures):
    try:
        df = pd.DataFrame([{
            'rating': features.rating,
            'metacritic': features.metacritic if features.metacritic is not None else 0,
            'playtime': features.playtime if features.playtime is not None else 0
        }])
        pred = model.predict(df)[0]
        prob = model.predict_proba(df)[0][1]
        return {
            "success": bool(pred),
            "probability": round(float(prob), 4),
            "message": "✅ Juego exitoso" if pred else "❌ Juego no exitoso"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Iniciar servidor en hilo separado
def start_api():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="critical")

threading.Thread(target=start_api, daemon=True).start()
print("\n🚀 FastAPI iniciada exitosamente")
print("   🔗 Accede a: http://localhost:8001/docs")
print("   ✅ Endpoints disponibles: /health, /predict")

📦 Cargando modelo XGBoost...
✅ Modelo cargado

🚀 FastAPI iniciada exitosamente
   🔗 Accede a: http://localhost:8000/docs
   ✅ Endpoints disponibles: /health, /predict
INFO:     127.0.0.1:62444 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:62444 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:50289 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:55718 - "POST /predict HTTP/1.1" 200 OK


In [52]:
# CELDA: IDENTIFICAR DIRECTORIO DEL PROYECTO
import os

print("📁 Directorio actual del notebook:")
print(f"   {os.getcwd()}")

print("\n📄 Archivos en este directorio:")
for f in os.listdir():
    if f.endswith(('.ipynb', '.md', '.py', '.json', '.csv')):
        print(f"   - {f}")

📁 Directorio actual del notebook:
   C:\Users\manue\proyecto_hackaboss\Proyecto_RAWG\rawg-aws-ml-analytics

📄 Archivos en este directorio:
   - config.py
   - main.ipynb
   - README.md


In [54]:
# CELDA: ACTUALIZAR README.md (versión segura)
import os

content = "# RAWG AWS ML Analytics\\n"
content += "Proyecto académico — Predicción de éxito de videojuegos\\n\\n"
content += "## Pipeline implementado\\n"
content += "✅ Lambda 1: Extracción histórica (2016–2026) → S3  \\n"
content += "✅ Lambda 2: Extracción incremental (últimos 7 días) → S3  \\n"
content += "✅ Lambda 3: Carga automática en RDS (PostgreSQL)  \\n"
content += "✅ Modelo ML: XGBoost con métrica `success = rating >= 4.0`  \\n"
content += "✅ FastAPI: Endpoints `/health`, `/predict`\\n\\n"
content += "## Métricas del modelo\\n"
content += "- Accuracy: 1.0000\\n"
content += "- ROC-AUC: 1.0000\\n"
content += "- Dataset: 133,295 juegos\\n\\n"
content += "## Estructura del proyecto\\n"
content += "```\n"
content += "rawg-aws-ml-analytics/\n"
content += "├── main.ipynb              # Notebook principal\n"
content += "├── README.md               # ← Este archivo\n"
content += "├── config.py               # Configuración de claves\n"
content += "├── data/\n"
content += "│   └── processed/\n"
content += "│       └── games_for_rds.csv\n"
content += "└── models/\n"
content += "    └── xgb_success_model.pkl\n"
content += "```\\n\\n"
content += "## Notas\\n"
content += "- Lambda 2 ejecutada el 2026-02-07 con rango de 7 días (10 juegos encontrados).\\n"
content += "- El pipeline está listo para automatización diaria mediante EventBridge.\\n"
content += "- Base de datos en RDS: rawg_games_db (133,295 registros)"

# Escribir archivo
with open("README.md", "w", encoding="utf-8") as f:
    f.write(content.replace("\\n", "\n"))

print("✅ README.md actualizado exitosamente")
print(f"   Ubicación: {os.path.abspath('README.md')}")

✅ README.md actualizado exitosamente
   Ubicación: C:\Users\manue\proyecto_hackaboss\Proyecto_RAWG\rawg-aws-ml-analytics\README.md


In [57]:
# CELDA 12A: Configuración para Anaconda
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio aplicado — FastAPI funcionará en Anaconda")

✅ nest_asyncio aplicado — FastAPI funcionará en Anaconda


In [59]:
# CELDA 12: FASTAPI COMPLETA (todos los endpoints en una sola celda)
# Puerto: 8001 (evita conflicto con puerto 8000)
# Incluye: /health, /predict, /games/{id}, /search

# Paso 1: Configuración para Anaconda (evita conflictos de event loop)
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio aplicado")

# Paso 2: Importar dependencias
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import joblib
import psycopg2
import pandas as pd
import threading
import uvicorn
import os
from datetime import datetime

# Paso 3: Configuración
MODEL_PATH = "models/xgb_success_model.pkl"
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

# Paso 4: Cargar modelo
print("📦 Cargando modelo XGBoost...")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Modelo no encontrado: {MODEL_PATH}")
model = joblib.load(MODEL_PATH)
print("✅ Modelo cargado")

# Paso 5: Crear aplicación FastAPI
app = FastAPI(
    title="RAWG Games Success Predictor API",
    description="API para predecir éxito de juegos usando XGBoost + datos de RDS",
    version="1.0.0"
)

# Paso 6: Modelos Pydantic
class GameFeatures(BaseModel):
    rating: float
    metacritic: Optional[float] = None
    playtime: Optional[int] = None

class PredictionResponse(BaseModel):
    success: bool
    probability: float
    message: str

class GameResponse(BaseModel):
    id: int
    name: str
    rating: Optional[float]
    metacritic: Optional[int]
    playtime: Optional[int]
    success: bool

class GameResult(BaseModel):
    id: int
    name: str
    released: Optional[str]
    rating: Optional[float]
    platforms: str
    success: bool

# Paso 7: Endpoint /health
@app.get("/health", summary="Verifica estado de la API")
def health():
    """Verifica que API, RDS y modelo estén operativos"""
    try:
        conn = psycopg2.connect(
            host=RDS_HOST, port=RDS_PORT, dbname=RDS_DBNAME,
            user=RDS_USER, password=RDS_PASSWORD
        )
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM games;")
        count = cur.fetchone()[0]
        cur.close()
        conn.close()
        return {
            "status": "healthy",
            "database": "connected",
            "games_count": count,
            "model_loaded": True
        }
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Database error: {str(e)}")

# Paso 8: Endpoint /predict
@app.post("/predict", response_model=PredictionResponse, summary="Predice éxito de un juego")
def predict(features: GameFeatures):
    """Predice si un juego será exitoso basado en sus características"""
    try:
        input_data = pd.DataFrame([{
            'rating': features.rating,
            'metacritic': features.metacritic if features.metacritic is not None else 0,
            'playtime': features.playtime if features.playtime is not None else 0
        }])
        
        prediction = model.predict(input_data)[0]
        probability = model.predict_proba(input_data)[0][1]
        message = "✅ El juego será exitoso" if prediction == 1 else "❌ El juego no será exitoso"
        
        return {
            "success": bool(prediction),
            "probability": round(float(probability), 4),
            "message": message
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")

# Paso 9: Endpoint /games/{game_id}
@app.get("/games/{game_id}", response_model=GameResponse, summary="Obtiene detalles de un juego por ID")
def get_game(game_id: int):
    """Obtiene información detallada de un juego específico"""
    try:
        conn = psycopg2.connect(
            host=RDS_HOST, port=RDS_PORT, dbname=RDS_DBNAME,
            user=RDS_USER, password=RDS_PASSWORD
        )
        cur = conn.cursor()
        cur.execute("""
            SELECT id, name, rating, metacritic, playtime, success
            FROM games
            WHERE id = %s;
        """, (game_id,))
        result = cur.fetchone()
        cur.close()
        conn.close()
        
        if result is None:
            raise HTTPException(status_code=404, detail=f"Juego con ID {game_id} no encontrado")
        
        return GameResponse(
            id=result[0],
            name=result[1],
            rating=result[2],
            metacritic=result[3],
            playtime=result[4],
            success=result[5]
        )
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error de base de datos: {str(e)}")

# Paso 10: Endpoint /search
@app.get("/search", response_model=List[GameResult], summary="Busca juegos por nombre")
def search_games(name: str, platform: Optional[str] = None):
    """
    Busca juegos por nombre (coincidencia parcial).
    Parámetros:
      - name: texto a buscar (ej: "cyberpunk")
      - platform: opcional (ej: "PC", "PlayStation 5")
    """
    try:
        conn = psycopg2.connect(
            host=RDS_HOST, port=RDS_PORT, dbname=RDS_DBNAME,
            user=RDS_USER, password=RDS_PASSWORD
        )
        cur = conn.cursor()
        
        query = """
            SELECT id, name, released, rating, success
            FROM games
            WHERE LOWER(name) LIKE LOWER(%s)
            ORDER BY rating DESC NULLS LAST
            LIMIT 10;
        """
        search_pattern = f"%{name}%"
        cur.execute(query, (search_pattern,))
        results = cur.fetchall()
        cur.close()
        conn.close()
        
        games = []
        for row in results:
            games.append(GameResult(
                id=row[0],
                name=row[1],
                released=row[2].strftime("%Y-%m-%d") if row[2] else None,
                rating=row[3],
                platforms="PC, PlayStation 5, Xbox Series X",  # Placeholder (datos limitados en tu esquema)
                success=row[4]
            ))
        
        return games
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error en búsqueda: {str(e)}")

# Paso 11: Iniciar servidor en hilo separado
def start_api():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="critical")

threading.Thread(target=start_api, daemon=True).start()
print("\n" + "="*60)
print("🚀 FASTAPI INICIADA EXITOSAMENTE")
print("="*60)
print("🔗 Accede a la documentación interactiva:")
print("   http://localhost:8001/docs")
print("\n✅ Endpoints disponibles:")
print("   GET  /health          → Estado del sistema")
print("   POST /predict         → Predice éxito de un juego")
print("   GET  /games/{id}      → Detalles de un juego por ID")
print("   GET  /search          → Busca juegos por nombre")
print("\n💡 Ejemplo de uso en /docs:")
print('   POST /predict → {"rating": 4.5, "metacritic": 85, "playtime": 100}')
print("="*60)

✅ nest_asyncio aplicado
📦 Cargando modelo XGBoost...
✅ Modelo cargado

🚀 FASTAPI INICIADA EXITOSAMENTE
🔗 Accede a la documentación interactiva:
   http://localhost:8001/docs

✅ Endpoints disponibles:
   GET  /health          → Estado del sistema
   POST /predict         → Predice éxito de un juego
   GET  /games/{id}      → Detalles de un juego por ID
   GET  /search          → Busca juegos por nombre

💡 Ejemplo de uso en /docs:
   POST /predict → {"rating": 4.5, "metacritic": 85, "playtime": 100}


In [64]:
# CELDA: CAPTURA #3 - CONTEO DE JUEGOS EN RDS
import psycopg2

# Configuración de RDS
RDS_HOST = "rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com"
RDS_PORT = 5432
RDS_USER = "adminuser"
RDS_PASSWORD = "Admin123456!"
RDS_DBNAME = "rawg_games_db"

# Conectar y ejecutar consulta
try:
    conn = psycopg2.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        dbname=RDS_DBNAME,
        user=RDS_USER,
        password=RDS_PASSWORD
    )
    cur = conn.cursor()
    
    # Ejecutar COUNT(*)
    cur.execute("SELECT COUNT(*) FROM games;")
    count = cur.fetchone()[0]
    
    # Cerrar conexión
    cur.close()
    conn.close()
    
    # Mostrar resultado
    print("✅ CAPTURA #3 - CONTEO DE JUEGOS EN RDS")
    print(f"📊 Total de juegos en la base de datos: {count:,}")
    print(f"🔗 Base de datos: {RDS_DBNAME}")
    print(f"📍 Host: {RDS_HOST}")
    
except Exception as e:
    print(f"❌ Error al conectar con RDS: {e}")

✅ CAPTURA #3 - CONTEO DE JUEGOS EN RDS
📊 Total de juegos en la base de datos: 133,295
🔗 Base de datos: rawg_games_db
📍 Host: rawg-rds-manuel-39.cvmk8iuyu1w5.eu-north-1.rds.amazonaws.com
